In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import explode
from pyspark.sql.functions import split
from pyspark.sql.types import StructType, StructField, StringType

#создаём SparkSession
spark = SparkSession.builder \
    .appName("count words") \
    .getOrCreate()

#определяем схему для датафрейма. Схема нужна, чтобы сформировать колонки с заданными именами и типами в датафрейме
#чтобы указать схему, используется конструкция StructType.
#чтобы указать имя колонки — StructField.
#типы колонки можно взять из pyspark.sql.types, в нашем случае нужен строковый тип StringType()
userSchema = StructType([StructField("text", StringType(), True)])

#читаем текст из файла
#применяем схему с помощью метода schema
#указываем, что читаем текстовый файл с помощью метода format('text')
#важно, что указываем в загрузке (метод load) именно директорию, а не файл, так как стриминг будет вычитывать все файлы из указанной директории
wordsDF = spark.readStream.schema(userSchema).format('text').load('/datas8')



#достаём из колонки text значение, разделяем слова по запятым, добавляем alias для колонки результата. Например, word
#(методы select, explode, split, alias)
splitWordsDF = wordsDF \
    .select(
        explode(split(wordsDF.test, ',')).alias('word')
    )

#группируем слова и считаем количество (методы groupBy, count)
wordCountsDF = splitWordsDF.groupBy('word').count

# запускаем стриминг
# важно указать outputMode("complete"), иначе по умолчанию будет выбран outputMode("append")
# вы считаете данные, поэтому при методе append данные уже не получится пересчитать, так как не получится дополнить результаты обработки данных
wordCountsDF.writeStream \
    .outputMode("complete") \
    .format("console") \
    .start() \
    .awaitTermination()